# Fresh Retail: Starter Notebook

This notebook accompanies the **Introduction** slide deck (`FreshRetail_Introduction.pptx`). It provides a shared data pipeline for both tracks, then produces the exact outputs previewed in the showcase slides.

**Run sections 1–4 first** (shared setup), then run the section for your track:

| Section | Track | What you produce |
|---------|-------|-----------------|
| 5. Operations | Ops | Temporal profiles, heatmaps, KPIs, hourly patterns |
| 6. Data Science | DS | WAPE baselines, forecast overlays, demand recovery, error analysis |

Both tracks use the same dataset, same helper functions, and same time split.

- **Operations Track**: O1 (Diagnosis) or O2 (Decision)
- **Data Science Track**: D1 (Direct benchmark) or D2 (Recovery first)

**Dataset**: [Dingdong-Inc/FreshRetailNet-50K](https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-50K)

---
## 1. Setup and Data Download

In [1]:
# Run this cell on Google Colab (already installed locally)
!pip install -q pandas pyarrow matplotlib seaborn datasets

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

print("Setup complete.")

Setup complete.


In [5]:
from datasets import load_dataset

print("Downloading FreshRetailNet-50K from Hugging Face...")
ds = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
print(ds)

# Convert to pandas
train_raw = ds["train"].to_pandas()
eval_raw = ds["eval"].to_pandas()

print(f"\nTrain: {train_raw.shape}, Eval: {eval_raw.shape}")
print(f"Columns: {list(train_raw.columns)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['city_id', 'store_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'product_id', 'dt', 'sale_amount', 'hours_sale', 'stock_hour6_22_cnt', 'hours_stock_status', 'discount', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level'],
        num_rows: 4500000
    })
    eval: Dataset({
        features: ['city_id', 'store_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'product_id', 'dt', 'sale_amount', 'hours_sale', 'stock_hour6_22_cnt', 'hours_stock_status', 'discount', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level'],
        num_rows: 350000
    })
})

Train: (4500000, 19), Eval: (350000, 19)
Columns: ['city_id', 'store_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'product_id', 'dt', 'sale_amount', 'hours_sale', 'sto

---
## 2. Data Preparation

In [6]:
def prepare_panel(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare the raw HF dataset into a clean analysis panel."""
    df = df.copy()

    # Parse date
    df["dt"] = pd.to_datetime(df["dt"])
    df = df.sort_values(["store_id", "product_id", "dt"]).reset_index(drop=True)

    # Create series_id (unique store x product combination)
    series_keys = df[["store_id", "product_id"]].drop_duplicates().reset_index(drop=True)
    series_keys["series_id"] = range(1, len(series_keys) + 1)
    df = df.merge(series_keys, on=["store_id", "product_id"], how="left")

    # Create day index (days since start)
    min_date = df["dt"].min()
    df["day_idx"] = (df["dt"] - min_date).dt.days + 1

    n_series = df["series_id"].nunique()
    n_days = df["day_idx"].nunique()
    print(f"Prepared {len(df):,} rows \u2014 {n_series:,} series x {n_days} days")
    print(f"Date range: {df['dt'].min().date()} to {df['dt'].max().date()}")
    return df


history = prepare_panel(train_raw)
history.head()

Prepared 4,500,000 rows — 50,000 series x 90 days
Date range: 2024-03-28 to 2024-06-25


,city_id,store_id,management_group_id,first_category_id,second_category_id,third_category_id,product_id,dt,sale_amount,hours_sale,...,hours_stock_status,discount,holiday_flag,activity_flag,precpt,avg_temperature,avg_humidity,avg_wind_level,series_id,day_idx
0,0,0,2,29,78,82,4,2024-03-28,0.5,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, ...",...,"[1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, ...",0.882,0,1,1.6999,15.48,73.54,1.97,1,1
1,0,0,2,29,78,82,4,2024-03-29,1.3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,"[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.882,0,1,3.0190,15.08,76.56,1.71,1,2
2,0,0,2,29,78,82,4,2024-03-30,5.3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.882,1,1,2.0942,15.91,76.47,1.73,1,3
3,0,0,2,29,78,82,4,2024-03-31,4.2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.879,1,1,1.5618,16.13,77.40,1.76,1,4
4,0,0,2,29,78,82,4,2024-04-01,0.7,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2, ...",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.882,0,1,3.5386,15.37,78.26,1.25,1,5


---
## 3. Shared Functions: flag_censoring, make_features, time_split

In [7]:
def flag_censoring(df: pd.DataFrame) -> pd.DataFrame:
    """Add censoring flags based on stockout hours."""
    df = df.copy()
    df["is_censored"] = (df["stock_hour6_22_cnt"] > 0).astype(int)
    df["censoring_severity"] = df["stock_hour6_22_cnt"] / 16
    print(f"Censored rows: {df['is_censored'].sum():,} / {len(df):,} ({df['is_censored'].mean():.1%})")
    return df


def make_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add lag and rolling features for EDA and forecasting."""
    df = df.sort_values(["series_id", "day_idx"]).copy()
    grp = df.groupby("series_id")["sale_amount"]
    df["sales_lag1"] = grp.shift(1)
    df["sales_lag7"] = grp.shift(7)
    df["sales_roll7"] = grp.transform(lambda x: x.rolling(7, min_periods=1).mean())
    df["sales_roll28"] = grp.transform(lambda x: x.rolling(28, min_periods=1).mean())
    df["psd"] = grp.transform("mean")  # per-series daily mean
    return df


def time_split(df: pd.DataFrame, horizon: int = 7) -> tuple:
    """Split into train and validation by time. Validation = last `horizon` days."""
    min_day = df["day_idx"].min()
    max_day = df["day_idx"].max()
    val_start = max_day - horizon + 1
    train = df[df["day_idx"] < val_start].copy()
    val = df[df["day_idx"] >= val_start].copy()
    print(f"Train: day {min_day}..{val_start - 1} ({len(train):,} rows), Val: day {val_start}..{max_day} ({len(val):,} rows)")
    return train, val

In [8]:
# Apply shared pipeline
history = flag_censoring(history)
history = make_features(history)

train, val = time_split(history, horizon=7)
print(f"\nValidation window: day {val['day_idx'].min()} to {val['day_idx'].max()}")

Censored rows: 1,992,006 / 4,500,000 (44.3%)
Train: day 1..83 (4,150,000 rows), Val: day 84..90 (350,000 rows)

Validation window: day 84 to 90


---
## 4. Data at a Glance

In [ ]:
# Show a real series with stockouts
series_stockouts = history.groupby("series_id")["is_censored"].mean()
example_sid = series_stockouts[(series_stockouts > 0.3) & (series_stockouts < 0.7)].index[0]

s_example = history[history["series_id"] == example_sid][
    ["dt", "day_idx", "sale_amount", "stock_hour6_22_cnt", "is_censored", "discount", "holiday_flag", "avg_temperature"]
].head(14)
print(f"Series {example_sid} \u2014 first 14 days (a product with frequent stockouts):")
display(s_example)

In [ ]:
# Dataset dimensions
summary = pd.Series({
    "Total rows": f"{len(history):,}",
    "Series (store x product)": f"{history['series_id'].nunique():,}",
    "Days per series": str(history["day_idx"].nunique()),
    "Products (product_id)": str(history["product_id"].nunique()),
    "Stores (store_id)": str(history["store_id"].nunique()),
    "Cities (city_id)": str(history["city_id"].nunique()),
    "Management groups": str(history["management_group_id"].nunique()),
    "Mean daily sales": f"{history['sale_amount'].mean():.3f}",
    "Censored rows": f"{history['is_censored'].sum():,} ({history['is_censored'].mean():.1%})",
    "Low-sale series (psd<1)": f"{(history.groupby('series_id')['psd'].first() < 1).sum():,}",
    "High-sale series (psd>=1)": f"{(history.groupby('series_id')['psd'].first() >= 1).sum():,}",
})
display(summary.to_frame("Value"))

In [ ]:
# Sales distribution and per-series daily mean
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

clipped = history["sale_amount"].clip(upper=history["sale_amount"].quantile(0.99))
axes[0].hist(clipped, bins=50, color="#065A82", edgecolor="white")
axes[0].set_title("Distribution of daily sales (clipped at 99th pctl)")
axes[0].set_xlabel("sale_amount")
axes[0].set_ylabel("Count")

psd_vals = history.groupby("series_id")["psd"].first()
axes[1].hist(psd_vals, bins=50, color="#1C7293", edgecolor="white")
axes[1].axvline(1.0, color="#E74C3C", linestyle="--", linewidth=2, label="psd=1 cutoff")
axes[1].set_title("Per-series daily mean (psd) distribution")
axes[1].set_xlabel("psd")
axes[1].set_ylabel("Number of series")
axes[1].legend()

plt.tight_layout()
plt.show()

---
---
# DATA SCIENCE TRACK

## 6. Data Science Track

> **You may skip this section if you are focusing on the Operations track.**

This section produces the following outputs — each corresponds to a slide in the deck:

| Output | What it shows | Slide |
|--------|--------------|-------|
| WAPE results table (D1) | Baseline comparison: global mean vs seasonal naive vs rolling 28d | Slide 18 |
| Forecast overlay chart | Predicted vs actual for one series across the validation window | Slide 18 |
| Recovery comparison table (D2) | WAPE on raw vs corrected target — does imputation help? | Slide 19 |
| WAPE by management group | Which product groups are hardest to forecast? | Slide 20 |
| Residual histogram + error scatter | Where the model fails and why | Slide 20 |

**A strong data science project** starts from these baselines and improves on them with better features, better imputation, or a more sophisticated model — always measured by WAPE on the same time split.

### 6a. WAPE Evaluation Function

In [9]:
def compute_wape(actual: np.ndarray, predicted: np.ndarray) -> float:
    """Weighted Absolute Percentage Error."""
    denom = np.sum(np.abs(actual))
    if denom == 0:
        return np.nan
    return np.sum(np.abs(actual - predicted)) / denom


def evaluate_forecast(val_df: pd.DataFrame, pred_col: str = "prediction") -> dict:
    """Compute WAPE overall, low-sale, high-sale, and harmonic mean.
    Only evaluates rows where stock_hour6_22_cnt == 0 (uncensored in validation)."""
    scored = val_df[val_df["stock_hour6_22_cnt"] == 0].copy()
    if len(scored) == 0:
        return {"wape_overall": np.nan}

    y = scored["sale_amount"].values
    yhat = scored[pred_col].values

    wape_all = compute_wape(y, yhat)

    low = scored[scored["psd"] < 1]
    high = scored[scored["psd"] >= 1]

    wape_low = compute_wape(low["sale_amount"].values, low[pred_col].values) if len(low) > 0 else np.nan
    wape_high = compute_wape(high["sale_amount"].values, high[pred_col].values) if len(high) > 0 else np.nan

    if np.isnan(wape_low) or np.isnan(wape_high) or wape_all == 0 or wape_low == 0 or wape_high == 0:
        hm = np.nan
    else:
        hm = 3 / (1/wape_all + 1/wape_low + 1/wape_high)

    return {
        "wape_overall": round(wape_all, 4) if not np.isnan(wape_all) else np.nan,
        "wape_low_sale": round(wape_low, 4) if not np.isnan(wape_low) else np.nan,
        "wape_high_sale": round(wape_high, 4) if not np.isnan(wape_high) else np.nan,
        "harmonic_mean": round(hm, 4) if not np.isnan(hm) else np.nan,
        "scored_rows": len(scored),
    }

print("Evaluation function ready.")

Evaluation function ready.


### 6b. D1 \u2014 Direct Benchmark: Naive Baselines on Raw Sales

In [ ]:
# --- Baseline 1: Global mean ---
series_mean = train.groupby("series_id")["sale_amount"].mean().rename("pred_global_mean")
val = val.drop(columns=["pred_global_mean", "pred_seasonal_naive", "pred_roll28", "forecast_day"], errors="ignore")
val = val.merge(series_mean, on="series_id", how="left")

# --- Baseline 2: Seasonal naive (last-week repeat) ---
val_start = val["day_idx"].min()
last_week = history[history["day_idx"].between(val_start - 7, val_start - 1)][["series_id", "day_idx", "sale_amount"]].copy()
last_week["forecast_day"] = last_week["day_idx"] + 7
last_week = last_week.rename(columns={"sale_amount": "pred_seasonal_naive"})

val = val.merge(last_week[["series_id", "forecast_day", "pred_seasonal_naive"]],
                left_on=["series_id", "day_idx"], right_on=["series_id", "forecast_day"], how="left")
val = val.drop(columns=["forecast_day"], errors="ignore")
val["pred_seasonal_naive"] = val["pred_seasonal_naive"].fillna(val["pred_global_mean"])

# --- Baseline 3: Rolling 28-day mean ---
roll28 = train.groupby("series_id")["sale_amount"].apply(
    lambda x: x.tail(28).mean(), include_groups=False
).rename("pred_roll28")
val = val.merge(roll28, on="series_id", how="left")

# Evaluate all three
results = {}
for method, col in [("Global mean", "pred_global_mean"), ("Seasonal naive", "pred_seasonal_naive"), ("Rolling 28d", "pred_roll28")]:
    val["prediction"] = val[col].clip(lower=0)
    results[method] = evaluate_forecast(val)

results_df = pd.DataFrame(results).T
print("=== D1 Benchmark Results ===")
display(results_df)

In [ ]:
# --- Visualize: forecast overlay for one series ---
example_sid3 = history.groupby("series_id")["psd"].first().sort_values(ascending=False).index[5]
ex = history[history["series_id"] == example_sid3].copy()
ex_val = val[val["series_id"] == example_sid3].copy()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ex["day_idx"], ex["sale_amount"], color="#065A82", linewidth=1.5, label="Actual (train)")
ax.plot(ex_val["day_idx"], ex_val["sale_amount"], color="#065A82", linewidth=2, linestyle="-", label="Actual (val)")
ax.plot(ex_val["day_idx"], ex_val["pred_global_mean"], color="#E67E22", linewidth=1.5, linestyle="--", label="Global mean")
ax.plot(ex_val["day_idx"], ex_val["pred_seasonal_naive"], color="#8E44AD", linewidth=1.5, linestyle="--", label="Seasonal naive")
ax.plot(ex_val["day_idx"], ex_val["pred_roll28"], color="#27AE60", linewidth=1.5, linestyle="--", label="Rolling 28d")

ax.axvline(ex_val["day_idx"].min() - 0.5, color="gray", linestyle=":", alpha=0.5)
ax.set_title(f"Series {example_sid3}: Forecast overlay (validation window)")
ax.set_xlabel("day_idx")
ax.set_ylabel("sale_amount")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 6c. D2 \u2014 Recovery First: Impute Censored Hours, Then Forecast

In [ ]:
# Expand hourly data from the list columns
# Note: this creates large arrays (~550MB). Colab free tier (12GB RAM) handles this fine.
print("Expanding hourly data...")

hourly_sales = np.stack(history["hours_sale"].values)          # (N, 24)
hourly_stock_ds = np.stack(history["hours_stock_status"].values)  # (N, 24)

# Focus on operating window h06..h21 (indices 6..21, 16 hours)
op_sales = hourly_sales[:, 6:22].astype(np.float32)
op_stock = hourly_stock_ds[:, 6:22].astype(np.float32)

# Mark censored hours
op_sales_masked = np.where(op_stock == 1, np.nan, op_sales)

total_cells = op_sales_masked.size
missing_cells = np.isnan(op_sales_masked).sum()
print(f"Operating window: {op_sales_masked.shape[1]} hours (h06-h21)")
print(f"Missing hourly cells: {missing_cells:,} / {total_cells:,} ({missing_cells/total_cells:.1%})")

In [ ]:
# --- Simple recovery: random pool sampling ---
visible_sum = np.nansum(np.where(op_stock == 0, op_sales, 0), axis=1)

imputed = op_sales_masked.copy()
imputed_count = 0
for h in range(16):
    col = imputed[:, h]
    mask = np.isnan(col)
    n_miss = mask.sum()
    if n_miss > 0:
        pool = col[~mask]
        imputed[mask, h] = np.maximum(0, rng.choice(pool, size=n_miss, replace=True))
        imputed_count += n_miss

# Rebuild corrected daily target
recovered_sum = np.nansum(imputed, axis=1)
outside_slice = np.maximum(history["sale_amount"].values.astype(np.float32) - visible_sum, 0)
recovered_daily = outside_slice + recovered_sum

history["recovered_daily_sales"] = recovered_daily

print(f"Imputed {imputed_count:,} hourly cells")
print(f"Mean raw sale_amount: {history['sale_amount'].mean():.4f}")
print(f"Mean recovered sales: {history['recovered_daily_sales'].mean():.4f}")

In [ ]:
# Re-split with recovered target
train_r, val_r = time_split(history, horizon=7)

# Seasonal naive on recovered target
val_start_r = val_r["day_idx"].min()
last_week_r = history[history["day_idx"].between(val_start_r - 7, val_start_r - 1)][
    ["series_id", "day_idx", "recovered_daily_sales"]
].copy()
last_week_r["forecast_day"] = last_week_r["day_idx"] + 7
last_week_r = last_week_r.rename(columns={"recovered_daily_sales": "pred_recovered_naive"})

val_r = val_r.merge(
    last_week_r[["series_id", "forecast_day", "pred_recovered_naive"]],
    left_on=["series_id", "day_idx"], right_on=["series_id", "forecast_day"], how="left",
)
val_r = val_r.drop(columns=["forecast_day"], errors="ignore")
fallback = train_r.groupby("series_id")["recovered_daily_sales"].mean()
val_r["pred_recovered_naive"] = val_r["pred_recovered_naive"].fillna(val_r["series_id"].map(fallback))

# Also add seasonal naive on raw for fair comparison
val_r = val_r.merge(
    val[["series_id", "day_idx", "pred_seasonal_naive"]].drop_duplicates(),
    on=["series_id", "day_idx"], how="left",
)

# Evaluate both
d2_results = {}
for method, col in [("Seasonal naive (raw)", "pred_seasonal_naive"), ("Seasonal naive (recovered)", "pred_recovered_naive")]:
    val_r["prediction"] = val_r[col].clip(lower=0)
    d2_results[method] = evaluate_forecast(val_r)

d2_df = pd.DataFrame(d2_results).T
print("=== D2 Recovery Comparison ===")
display(d2_df)

## 6f. D2 - 3 recovering strategy


### per-series mean



In [10]:
# LGBMRegressor
from lightgbm import LGBMRegressor
# drop missing value
train_Lgbm = train.dropna()
val_Lgbm = val.dropna()

features = [
    "sales_lag1",
    "sales_lag7",
    "sales_roll7",
    "sales_roll28",
    "avg_temperature",
    'management_group_id',
    'city_id',
    'product_id',
    'holiday_flag',
    'is_censored',
    'discount']

target = "sale_amount"

model = LGBMRegressor(random_state=1)
model.fit(train_Lgbm[features], train_Lgbm[target])

val_Lgbm["prediction"] = model.predict(val_Lgbm[features])
val_Lgbm["prediction"] = val_Lgbm["prediction"].clip(lower=0)

evaluate_forecast(val_Lgbm)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.179927 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1763
[LightGBM] [Info] Number of data points in the train set: 3800000, number of used features: 11
[LightGBM] [Info] Start training from score 0.999225


{'wape_overall': np.float64(0.2561),
 'wape_low_sale': np.float64(0.3142),
 'wape_high_sale': np.float64(0.2111),
 'harmonic_mean': np.float64(0.2537),
 'scored_rows': 198987}

### LGBMRegressor with Per-Series Hourly Mean Recovered Data

In [ ]:
print("Implementing Per-Series Hourly Mean recovery (leakage-free)...")

# --- Start: Necessary definitions moved from fyqpkO6BPLFc to ensure scope ---
# Expand hourly data from the list columns
# Note: this creates large arrays (~550MB). Colab free tier (12GB RAM) handles this fine.
print("Expanding hourly data...")

hourly_sales = np.stack(history["hours_sale"].values)          # (N, 24)
hourly_stock_ds = np.stack(history["hours_stock_status"].values)  # (N, 24)

# Focus on operating window h06..h21 (indices 6..21, 16 hours)
op_sales = hourly_sales[:, 6:22].astype(np.float32)
op_stock = hourly_stock_ds[:, 6:22].astype(np.float32)

# Mask censored hours
op_sales_masked = np.where(op_stock == 1, np.nan, op_sales)

total_cells = op_sales_masked.size
missing_cells = np.isnan(op_sales_masked).sum()
print(f"Operating window: {op_sales_masked.shape[1]} hours (h06-h21)")
print(f"Missing hourly cells: {missing_cells:,} / {total_cells:,} ({missing_cells/total_cells:.1%})")
# --- End: Necessary definitions moved ---


# Determine the validation start day from the original time_split for training data
# This ensures the means are calculated only from the training period
val_start_pshm_calc = history['day_idx'].max() - 7 + 1 # horizon is 7

# Filter history to create a training-only view for mean calculation
train_history_for_pshm_calc = history[history["day_idx"] < val_start_pshm_calc].copy()

# Calculate per-series hourly means FOR TRAINING DATA ONLY
per_series_hourly_mean_data = {}

# Iterate through unique series_id present in the training data
train_series_ids = train_history_for_pshm_calc['series_id'].unique()

for series_id in train_series_ids:
    # Get the original history indices that correspond to this series_id AND are in the training set
    original_history_indices_for_series_in_train = train_history_for_pshm_calc[train_history_for_pshm_calc['series_id'] == series_id].index

    # Extract the operating window sales and stock status for this series
    # from the FULL op_sales/op_stock arrays, but only for the training period rows.
    series_op_sales_train = op_sales[original_history_indices_for_series_in_train, :]
    series_op_stock_train = op_stock[original_history_indices_for_series_in_train, :]

    # Mask sales for stockouts within this series' training data
    series_op_sales_masked_train = np.where(series_op_stock_train == 1, np.nan, series_op_sales_train)

    # Initialize hourly_means with NaN for all hours
    hourly_means = np.full(series_op_sales_masked_train.shape[1], np.nan)

    # Calculate the mean for each hour for this specific series, ignoring NaNs
    # This mean is based ONLY on training data for this series, avoiding leakage.
    # Explicitly check for all-NaN slices to prevent RuntimeWarning
    for hour_idx in range(series_op_sales_masked_train.shape[1]):
        hour_sales_data = series_op_sales_masked_train[:, hour_idx]
        if not np.all(np.isnan(hour_sales_data)):
            hourly_means[hour_idx] = np.nanmean(hour_sales_data)

    per_series_hourly_mean_data[series_id] = hourly_means

# Impute using per-series hourly mean (apply to the full masked data using means from training)
imputed_pshm = op_sales_masked.copy() # Start with the full masked data

imputed_count_pshm = 0

for i in range(len(history)): # Iterate over each series-day observation in the full history
    current_series_id = history['series_id'].iloc[i]

    # Get the pre-calculated hourly means for this series (derived from training data)
    series_hourly_means = per_series_hourly_mean_data.get(current_series_id)

    if series_hourly_means is not None: # Means exist for this series from training data
        for h in range(16): # Iterate over each hour in the operating window
            if np.isnan(imputed_pshm[i, h]):
                mean_val = series_hourly_means[h]
                if not np.isnan(mean_val): # Only impute if a mean exists for this series-hour
                    imputed_pshm[i, h] = np.maximum(0, mean_val)
                    imputed_count_pshm += 1
                else:
                    # Fallback if specific series-hour mean is NaN (e.g., this series always censored for this hour in training)
                    imputed_pshm[i, h] = 0
    else:
        # If a series_id is in history but not in train_series_ids (e.g., it only appears in validation),
        # its hourly_means won't be in per_series_hourly_mean_data. Impute with 0 in this case.
        for h in range(16):
            if np.isnan(imputed_pshm[i, h]):
                imputed_pshm[i, h] = 0


# Rebuild corrected daily target (using the imputed_pshm which now covers the full history)
# visible_sum is from the original op_sales (non-censored hours)
visible_sum_pshm = np.nansum(np.where(op_stock == 0, op_sales, 0), axis=1)
recovered_sum_pshm = np.sum(imputed_pshm, axis=1) # Sum of the (now) fully imputed hourly sales

# outside_slice is sales outside operating hours, adjusted to be non-negative
outside_slice_pshm = np.maximum(history["sale_amount"].values.astype(np.float32) - visible_sum_pshm, 0)
recovered_daily_pshm = outside_slice_pshm + recovered_sum_pshm

history["recovered_daily_sales_pshm"] = recovered_daily_pshm

print(f"Imputed {imputed_count_pshm:,} hourly cells using per-series hourly mean (training data only).")
print(f"Mean raw sale_amount: {history['sale_amount'].mean():.4f}")
print(f"Mean recovered sales (per-series hourly mean): {history['recovered_daily_sales_pshm'].mean():.4f}")

# Re-split with recovered target (this will now use the newly updated history)
train_r_pshm, val_r_pshm = time_split(history, horizon=7)

# LGBMRegressor on per-series hourly mean recovered sales
from lightgbm import LGBMRegressor

# Drop missing values from the train_r_pshm and val_r_pshm dataframes
# (these were already created from history with 'recovered_daily_sales_pshm')
train_Lgbm_pshm = train_r_pshm.dropna()
val_Lgbm_pshm = val_r_pshm.dropna()

features_lgbm = [
    "sales_lag1",
    "sales_lag7",
    "sales_roll7",
    "sales_roll28",
    "avg_temperature",
    'management_group_id',
    'city_id',
    'product_id',
    'holiday_flag',
    'is_censored',
    'discount'
]

target_lgbm_pshm = "recovered_daily_sales_pshm"

model_pshm = LGBMRegressor(random_state=1)
model_pshm.fit(train_Lgbm_pshm[features_lgbm], train_Lgbm_pshm[target_lgbm_pshm])

val_Lgbm_pshm["prediction"] = model_pshm.predict(val_Lgbm_pshm[features_lgbm])
val_Lgbm_pshm["prediction"] = val_Lgbm_pshm["prediction"].clip(lower=0)

print("=== LGBMRegressor on Per-Series Hourly Mean Recovered Data (Leakage-Free) ===")
# When evaluating, 'sale_amount' should be the actual target, which is recovered_daily_sales_pshm
val_Lgbm_pshm_eval = val_Lgbm_pshm.copy()
val_Lgbm_pshm_eval["sale_amount"] = val_Lgbm_pshm_eval[target_lgbm_pshm]

pshm_lgbm_results = evaluate_forecast(val_Lgbm_pshm_eval)
print(pshm_lgbm_results)

# If all_recovery_results is not in scope, this would need re-initialization or a new comparison table
# For now, let's just display it and assume it can be merged later.


Implementing Per-Series Hourly Mean recovery (leakage-free)...
Expanding hourly data...
Operating window: 16 hours (h06-h21)
Missing hourly cells: 14,311,536 / 72,000,000 (19.9%)


### Weighted Sampling by Recency Recovery Strategy

In [ ]:
print("Implementing Weighted Sampling by Recency recovery (leakage-free)...")

imputed_wsr = op_sales_masked.copy()
imputed_count_wsr = 0

# Create a function to get weighted choices
def weighted_choice(data, weights, k):
    if len(data) == 0:
        return np.array([])
    # Normalize weights
    norm_weights = weights / np.sum(weights)
    return rng.choice(data, size=k, p=norm_weights, replace=True)

# Determine the validation start day from the original time_split for training data
# This ensures the sampling is based only on data available BEFORE the validation period
val_start_wsr_calc = history['day_idx'].max() - 7 + 1 # horizon is 7

for i in range(len(history)): # Iterate over each series-day observation in the full history
    current_series_id = history['series_id'].iloc[i]
    current_day_idx = history['day_idx'].iloc[i]

    # Filter history to consider only days *before* current_day_idx AND *before* the validation period starts
    # This ensures no leakage from future or validation data
    relevant_days_for_sampling_indices = history[
        (history['series_id'] == current_series_id) &
        (history['day_idx'] < current_day_idx) &
        (history['day_idx'] < val_start_wsr_calc)
    ].index

    for h in range(16): # Iterate over each hour in the operating window
        if np.isnan(imputed_wsr[i, h]):
            if len(relevant_days_for_sampling_indices) > 0:
                # Get the sales and stock status for the relevant hour for these days
                # Need to use the original `hourly_sales` and `hourly_stock_ds` arrays for this
                potential_sales = hourly_sales[relevant_days_for_sampling_indices, h + 6] # h+6 to map back to 24-hour index
                potential_stock_status = hourly_stock_ds[relevant_days_for_sampling_indices, h + 6]

                # Filter for non-censored sales at this specific hour
                observed_sales_pool = potential_sales[potential_stock_status == 0]
                observed_days_indices = relevant_days_for_sampling_indices[potential_stock_status == 0]

                if len(observed_sales_pool) > 0:
                    # Calculate recency weights for the observed sales
                    recency_diff = current_day_idx - history['day_idx'].iloc[observed_days_indices]
                    # Using inverse of recency_diff (or similar) as weight. Add small epsilon to avoid division by zero.
                    weights = 1 / (recency_diff + 1e-6)

                    # Sample one value for the missing hour
                    imputed_val = weighted_choice(observed_sales_pool, weights, 1)[0]
                    imputed_wsr[i, h] = np.maximum(0, imputed_val)
                    imputed_count_wsr += 1
                else:
                    # No observed sales for this series and hour in the past (within training), impute with 0
                    imputed_wsr[i, h] = 0
            else:
                # No previous days for this series (within training) to sample from, impute with 0
                imputed_wsr[i, h] = 0

# Rebuild corrected daily target for WSR
# visible_sum is the sum of sales from non-censored hours (original logic)
visible_sum_wsr = np.nansum(np.where(op_stock == 0, op_sales, 0), axis=1)
recovered_sum_wsr = np.sum(imputed_wsr, axis=1) # Sum of the (now) fully imputed hourly sales

# outside_slice is sales outside operating hours, adjusted to be non-negative
outside_slice_wsr = np.maximum(history["sale_amount"].values.astype(np.float32) - visible_sum_wsr, 0)
recovered_daily_wsr = outside_slice_wsr + recovered_sum_wsr

history["recovered_daily_sales_wsr"] = recovered_daily_wsr

print(f"Imputed {imputed_count_wsr:,} hourly cells using weighted sampling by recency (training data only).")
print(f"Mean raw sale_amount: {history['sale_amount'].mean():.4f}")
print(f"Mean recovered sales (weighted sampling by recency): {history['recovered_daily_sales_wsr'].mean():.4f}")

# Re-split with recovered target (this will now use the newly updated history)
train_r_wsr, val_r_wsr = time_split(history, horizon=7)

# LGBMRegressor on weighted sampling by recency recovered sales
from lightgbm import LGBMRegressor

# Drop missing values from the train_r_wsr and val_r_wsr dataframes
train_Lgbm_wsr = train_r_wsr.dropna()
val_Lgbm_wsr = val_r_wsr.dropna()

features_lgbm = [
    "sales_lag1",
    "sales_lag7",
    "sales_roll7",
    "sales_roll28",
    "avg_temperature",
    'management_group_id',
    'city_id',
    'product_id',
    'holiday_flag',
    'is_censored',
    'discount'
]

target_lgbm_wsr = "recovered_daily_sales_wsr"

model_wsr = LGBMRegressor(random_state=1)
model_wsr.fit(train_Lgbm_wsr[features_lgbm], train_Lgbm_wsr[target_lgbm_wsr])

val_Lgbm_wsr["prediction"] = model_wsr.predict(val_Lgbm_wsr[features_lgbm])
val_Lgbm_wsr["prediction"] = val_Lgbm_wsr["prediction"].clip(lower=0)

print("=== LGBMRegressor on Weighted Sampling by Recency Recovered Data (Leakage-Free) ===")
# When evaluating, 'sale_amount' should be the actual target, which is recovered_daily_sales_wsr
val_Lgbm_wsr_eval = val_Lgbm_wsr.copy()
val_Lgbm_wsr_eval["sale_amount"] = val_Lgbm_wsr_eval[target_lgbm_wsr]

wsr_lgbm_results = evaluate_forecast(val_Lgbm_wsr_eval)
print(wsr_lgbm_results)

# Add to the all_recovery_results for final comparison (assuming it was initialized earlier)
# If all_recovery_results is not in scope, this would need re-initialization or a new comparison table
# For now, let's just display it and assume it can be merged later.


### LGBMRegressor with Weighted Sampling by Recency Recovered Data

In [ ]:
# LGBMRegressor on weighted sampling by recency recovered sales
from lightgbm import LGBMRegressor

# Drop missing values from the train_r_wsr and val_r_wsr dataframes
train_Lgbm_wsr = train_r_wsr.dropna()
val_Lgbm_wsr = val_r_wsr.dropna()

features_lgbm = [
    "sales_lag1",
    "sales_lag7",
    "sales_roll7",
    "sales_roll28",
    "avg_temperature",
    'management_group_id',
    'city_id',
    'product_id',
    'holiday_flag',
    'is_censored',
    'discount'
]

target_lgbm_wsr = "recovered_daily_sales_wsr"

model_wsr = LGBMRegressor(random_state=1)
model_wsr.fit(train_Lgbm_wsr[features_lgbm], train_Lgbm_wsr[target_lgbm_wsr])

val_Lgbm_wsr["prediction"] = model_wsr.predict(val_Lgbm_wsr[features_lgbm])
val_Lgbm_wsr["prediction"] = val_Lgbm_wsr["prediction"].clip(lower=0)

print("=== LGBMRegressor on Weighted Sampling by Recency Recovered Data ===")
# When evaluating, 'sale_amount' should be the actual target, which is recovered_daily_sales_wsr
val_Lgbm_wsr_eval = val_Lgbm_wsr.copy()
val_Lgbm_wsr_eval["sale_amount"] = val_Lgbm_wsr_eval[target_lgbm_wsr]

wsr_lgbm_results = evaluate_forecast(val_Lgbm_wsr_eval)
print(wsr_lgbm_results)

# Add to the all_recovery_results for final comparison (assuming it was initialized earlier)
# If all_recovery_results is not in scope, this would need re-initialization or a new comparison table
# For now, let's just display it and assume it can be merged later.


### 6d. Error Analysis

In [ ]:
# WAPE by management group
scored = val[val["stock_hour6_22_cnt"] == 0].copy()
scored["prediction"] = scored["pred_seasonal_naive"].clip(lower=0)
scored["abs_error"] = np.abs(scored["sale_amount"] - scored["prediction"])

group_wape = scored.groupby("management_group_id").apply(
    lambda g: compute_wape(g["sale_amount"].values, g["prediction"].values),
    include_groups=False
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
group_wape.plot(kind="barh", color="#1C7293", edgecolor="white", ax=ax)
ax.set_title("WAPE by management group (seasonal naive baseline)")
ax.set_xlabel("WAPE")
ax.set_ylabel("Management Group ID")
plt.tight_layout()
plt.show()

In [ ]:
# Residual distribution and error vs stockout frequency
scored["residual"] = scored["sale_amount"] - scored["prediction"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(scored["residual"].clip(-5, 5), bins=60, color="#065A82", edgecolor="white")
axes[0].axvline(0, color="#E74C3C", linestyle="--", linewidth=2)
axes[0].set_title("Residual distribution (seasonal naive)")
axes[0].set_xlabel("Actual - Predicted")
axes[0].set_ylabel("Count")

series_error = scored.groupby("series_id").agg(
    mean_abs_error=("abs_error", "mean"),
).reset_index()
series_error = series_error.merge(
    history.groupby("series_id")["is_censored"].mean().rename("stockout_freq"),
    on="series_id"
)
axes[1].scatter(series_error["stockout_freq"], series_error["mean_abs_error"], alpha=0.1, s=5, color="#065A82")
axes[1].set_title("Mean absolute error vs stockout frequency")
axes[1].set_xlabel("Stockout frequency")
axes[1].set_ylabel("Mean absolute error")

plt.tight_layout()
plt.show()

---
---
## 7. Next Steps

### Operations Track

**O1 \u2014 Diagnosis First**
- Extend the heatmaps to find which store x category combinations are most fragile
- Test whether promotions (`discount < 1`) increase late-day stockouts
- Run panel regressions with fixed effects to isolate drivers

**O2 \u2014 Decision First**
- Build a simple corrected demand estimate (impute censored hours from `hours_sale`)
- Compute newsvendor order quantities under raw vs. corrected demand
- Visualize the service vs. waste trade-off curve

### Data Science Track

**D1 \u2014 Direct Benchmark**
- Try exponential smoothing or a simple LightGBM with lag features
- Analyze errors by day-of-week to detect weekly patterns
- Compare WAPE across cities to find geographic patterns

**D2 \u2014 Recovery First**
- Try per-series mean imputation instead of global pool sampling
- Compare multiple recovery strategies on the same baseline
- Focus error analysis on high-stockout series where recovery matters most

### Cross-Track Synergies
- Operations insights (which products are most fragile) can inform DS feature engineering
- DS demand recovery estimates can feed back into operations policy evaluation
- Both tracks benefit from understanding the hourly censoring structure